## 0. Setup

In [ ]:
import os

import torch, torch.nn as nn, torch.nn.functional as F
torch.backends.cudnn.enabled = False  # fix LSTM CUDA kernel error
import pandas as pd, numpy as np
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from transformers import BertModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt, seaborn as sns
from tqdm.auto import tqdm
import warnings, random, time
warnings.filterwarnings('ignore')

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)

def build_runtime_note(device, note):
    lines = [f'device: {device}', f'torch: {torch.__version__}', note]
    lower = note.lower()
    if device.type == 'cpu' and ('no kernel image' in lower or 'cuda fallback' in lower):
        lines.append('GPU advice: current CUDA/PyTorch build is incompatible with this GPU.')
        lines.append('Recommended on Kaggle: switch accelerator to T4, L4, or A100.')
        lines.append('If you must use P100, install a torch build that supports sm_60.')
    return ' | '.join(lines)

def pick_device():
    if os.environ.get('FORCE_CPU', '0') == '1':
        return torch.device('cpu'), 'FORCE_CPU=1'
    if not torch.cuda.is_available():
        return torch.device('cpu'), 'CUDA not available'
    try:
        # Small CUDA op to verify this GPU can run the installed torch CUDA kernels.
        x = torch.tensor([1.0], device='cuda')
        _ = x * 2.0
        torch.cuda.synchronize()
        name = torch.cuda.get_device_name(0)
        cc = torch.cuda.get_device_capability(0)
        gpu_note = f'{name} (cc {cc[0]}.{cc[1]})'
        if cc[0] < 7:
            gpu_note += ' | Note: older GPU arch; newer CUDA wheels may fail on some setups.'
        return torch.device('cuda'), gpu_note
    except Exception as e:
        return torch.device('cpu'), f'CUDA fallback: {type(e).__name__}: {e}'

set_seed(42)
device, device_note = pick_device()
print(build_runtime_note(device, device_note))


## 1. BERT + BiLSTM Model

In [ ]:
class EnhancedBertForSequenceClassification(nn.Module):
    def __init__(self, model_name='bert-base-uncased', num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = BertModel.from_pretrained(model_name, low_cpu_mem_usage=False, device_map=None)
        self.dropout = nn.Dropout(dropout)
        h = self.bert.config.hidden_size         # 768
        self.lstm = nn.LSTM(h, 256, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
        self.attention = nn.MultiheadAttention(512, num_heads=8, dropout=0.1, batch_first=True)
        self.layer_norm = nn.LayerNorm(512)
        self.classifier = nn.Sequential(
            nn.Linear(512,256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256,128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        seq = self.dropout(self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state)
        out = self.layer_norm(self.lstm(seq)[0])
        out, _ = self.attention(out, out, out)
        return self.classifier(torch.max(out, dim=1).values)

print("Model defined ✅")


## 2. Dataset Class

In [ ]:
class WELFakeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts, self.labels, self.tokenizer, self.max_length = texts, labels, tokenizer, max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding='max_length',
                             max_length=self.max_length, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(),
                'attention_mask': enc['attention_mask'].flatten(),
                'labels': torch.tensor(self.labels[idx], dtype=torch.long)}
print("Dataset defined ✅")


## 3. Load & Preprocess Data

In [ ]:
DATA_PATH = "/kaggle/input/datasets/kfsurya/welfake/preprocessed_welfake.csv"
df = pd.read_csv(DATA_PATH)
if 'Unnamed: 0' in df.columns: df.drop('Unnamed: 0', axis=1, inplace=True)
df['title'] = df['title'].fillna('').astype(str)
df['text']  = df['text'].fillna('').astype(str)
df['enhanced_text'] = df['title'] + ' [SEP] ' + df['text'].str[:1000]
df['encoded_label'] = df['label'].astype(int)
df = df[df['enhanced_text'].str.len() > 10].reset_index(drop=True)

train_df, temp_df = train_test_split(df, test_size=0.30, random_state=42, stratify=df['encoded_label'])
val_df, test_df   = train_test_split(temp_df, test_size=0.50, random_state=42, stratify=temp_df['encoded_label'])
print(f"Train {len(train_df)} | Val {len(val_df)} | Test {len(test_df)}")


## 4. Config + DataLoaders

In [ ]:
CONFIG = {
    'model_name': 'bert-base-uncased', 'max_length': 512,
    'batch_size': 8, 'learning_rate': 2e-5, 'epochs': 25,
    'dropout': 0.3, 'warmup_ratio': 0.1, 'weight_decay': 0.01,
    'gradient_accumulation_steps': 4, 'patience': 3,
    'freeze_layers': 6, 'label_smoothing': 0.1,
}

tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])

# --- CPU mode overrides keep runtime manageable when CUDA is unavailable/incompatible ---
_on_cpu = (device.type == 'cpu')
if _on_cpu:
    CONFIG.update({
        'max_length': 256,
        'batch_size': 4,
        'epochs': 5,
        'gradient_accumulation_steps': 2,
    })
    print('CPU mode overrides applied:', {
        'max_length': CONFIG['max_length'],
        'batch_size': CONFIG['batch_size'],
        'epochs': CONFIG['epochs'],
        'gradient_accumulation_steps': CONFIG['gradient_accumulation_steps'],
    })

# --- CPU subset: smaller data when no GPU so runtime stays manageable ---
if _on_cpu:
    train_df = train_df.sample(n=min(5000, len(train_df)), random_state=42).reset_index(drop=True)
    val_df   = val_df.sample(n=min(1000,  len(val_df)),   random_state=42).reset_index(drop=True)
    print(f'CPU subset → Train: {len(train_df)}  Val: {len(val_df)}')

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced',
        classes=np.unique(train_df['encoded_label']),
        y=train_df['encoded_label'])
class_weights = torch.tensor(cw, dtype=torch.float).to(device)
print(f'Class weights: Real={cw[0]:.3f}  Fake={cw[1]:.3f}')

train_ds = WELFakeDataset(train_df['enhanced_text'].values, train_df['encoded_label'].values, tokenizer, CONFIG['max_length'])
val_ds   = WELFakeDataset(val_df['enhanced_text'].values,   val_df['encoded_label'].values,   tokenizer, CONFIG['max_length'])

from torch.utils.data import WeightedRandomSampler
sampler = WeightedRandomSampler([cw[l] for l in train_df['encoded_label']], len(train_df), replacement=True)
# num_workers>0 and pin_memory=True only help on GPU; on CPU/Windows they cause overhead
_nw = 0 if _on_cpu else 2
_pm = not _on_cpu
train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], sampler=sampler, num_workers=_nw, pin_memory=_pm)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False,  num_workers=_nw, pin_memory=_pm)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')
print(f'Effective batch size: {CONFIG["batch_size"] * CONFIG["gradient_accumulation_steps"]}')


## 5. Training & Eval Functions

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device, accum):
    model.train(); total_loss = correct = total = 0; optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc='Training')):
        ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
        lbls = batch['labels'].to(device)
        # Single forward pass - reuse logits for both loss and accuracy
        logits = model(input_ids=ids, attention_mask=mask)
        loss = criterion(logits, lbls) / accum
        loss.backward()
        if (step+1) % accum == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
        total_loss += loss.item()*accum
        correct += (logits.detach().argmax(1)==lbls).sum().item(); total += lbls.size(0)
    return total_loss/len(loader), correct/total

def evaluate(model, loader, criterion, device):
    model.eval(); total_loss=0; preds, labels=[], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Evaluating'):
            ids=batch['input_ids'].to(device); mask=batch['attention_mask'].to(device)
            lbls=batch['labels'].to(device)
            out=model(input_ids=ids, attention_mask=mask)
            total_loss+=criterion(out,lbls).item()
            preds.extend(out.argmax(1).cpu().numpy()); labels.extend(lbls.cpu().numpy())
    f1=f1_score(labels,preds,average='weighted')
    p,r,*_=precision_recall_fscore_support(labels,preds,average='weighted')
    return dict(loss=total_loss/len(loader), accuracy=accuracy_score(labels,preds), f1=f1, precision=p, recall=r)

print("Functions defined ✅")


## 6. Training Loop

In [ ]:
model = EnhancedBertForSequenceClassification(
    CONFIG['model_name'], num_classes=2, dropout=CONFIG['dropout']
).to(device)

for p in model.bert.embeddings.parameters(): p.requires_grad = False
for i, layer in enumerate(model.bert.encoder.layer):
    if i < CONFIG['freeze_layers']:
        for p in layer.parameters(): p.requires_grad = False

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), label_smoothing=CONFIG['label_smoothing'])
optimizer = AdamW(model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay'])
total_steps = len(train_loader)*CONFIG['epochs']//CONFIG['gradient_accumulation_steps']
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps*CONFIG['warmup_ratio']), total_steps)

history = {k:[] for k in ['train_loss','train_acc','val_loss','val_acc','val_f1','val_precision','val_recall']}
best_f1, best_state, patience_ctr = 0.0, None, 0

for epoch in range(CONFIG['epochs']):
    print(f"\n📍 Epoch {epoch+1}/{CONFIG['epochs']}")
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, scheduler, criterion, device, CONFIG['gradient_accumulation_steps'])
    val = evaluate(model, val_loader, criterion, device)
    for k,v in [('train_loss',tr_loss),('train_acc',tr_acc),('val_loss',val['loss']),
                ('val_acc',val['accuracy']),('val_f1',val['f1']),
                ('val_precision',val['precision']),('val_recall',val['recall'])]:
        history[k].append(v)
    print(f"  Train loss={tr_loss:.4f} acc={tr_acc:.4f}")
    print(f"  Val   loss={val['loss']:.4f} acc={val['accuracy']:.4f} F1={val['f1']:.4f}")
    if val['f1'] > best_f1:
        best_f1=val['f1']; best_state={k:v.cpu() for k,v in model.state_dict().items()}
        patience_ctr=0; print(f"  🏆 Best F1: {best_f1:.4f}")
    else:
        patience_ctr+=1
        if patience_ctr>=CONFIG['patience']: print("🛑 Early stopping"); break

model.load_state_dict({k:v.to(device) for k,v in best_state.items()})
print(f"\n✅ Done | Best Val F1: {best_f1:.4f}")


## 7. Training Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ep = range(1, len(history['train_loss'])+1)
axes[0].plot(ep, history['train_loss'],'b-o',label='Train'); axes[0].plot(ep, history['val_loss'],'r-o',label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=.3)
axes[1].plot(ep, history['train_acc'],'b-o',label='Train'); axes[1].plot(ep, history['val_acc'],'r-o',label='Val')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=.3)
axes[2].plot(ep, history['val_f1'],'g-o',label='F1')
axes[2].plot(ep, history['val_precision'],'m-s',label='Precision')
axes[2].plot(ep, history['val_recall'],'y-^',label='Recall')
axes[2].set_title('Val Metrics'); axes[2].legend(); axes[2].grid(alpha=.3)
plt.tight_layout(); plt.savefig('training_history.png',dpi=120,bbox_inches='tight'); plt.show()


## 8. Traditional ML Baselines

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer

# Use full training data for ML models (they train fast)
vec   = TfidfVectorizer(max_features=30000, sublinear_tf=True, ngram_range=(1,2))
X_tr  = vec.fit_transform(train_df['enhanced_text'])
X_val = vec.transform(val_df['enhanced_text'])
y_tr  = train_df['encoded_label'].values
y_val = val_df['encoded_label'].values

ml_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Linear SVC':          LinearSVC(max_iter=2000, random_state=42),
    'Naive Bayes':         MultinomialNB(),
}

results = []
for name, clf in ml_models.items():
    t0=time.time(); clf.fit(X_tr, y_tr); preds=clf.predict(X_val)
    p,r,*_=precision_recall_fscore_support(y_val,preds,average='weighted')
    results.append(dict(Model=name, Accuracy=accuracy_score(y_val,preds),
                        F1=f1_score(y_val,preds,average='weighted'),
                        Precision=p, Recall=r))
    print(f"{name:22s}  Acc={results[-1]['Accuracy']:.4f}  F1={results[-1]['F1']:.4f}  ({time.time()-t0:.1f}s)")

results.append(dict(Model='BERT+BiLSTM ⭐',
    Accuracy=history['val_acc'][-1], F1=history['val_f1'][-1],
    Precision=history['val_precision'][-1], Recall=history['val_recall'][-1]))

results_df = pd.DataFrame(results)
print("\n", results_df[['Model','Accuracy','F1','Precision','Recall']].to_string(index=False))


## 9. Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
melted = results_df.melt(id_vars='Model', value_vars=['Accuracy','F1','Precision','Recall'],
                         var_name='Metric', value_name='Score')
sns.barplot(data=melted, x='Metric', y='Score', hue='Model', ax=axes[0], palette='viridis')
axes[0].set_title('All Metrics', fontsize=13, fontweight='bold')
axes[0].set_ylim(0.4, 1.05); axes[0].legend(bbox_to_anchor=(1.05,1), loc='upper left')
axes[0].grid(axis='y', alpha=0.3)

sdf = results_df.sort_values('Accuracy')
bars = axes[1].barh(sdf['Model'], sdf['Accuracy'], color=sns.color_palette('viridis', len(sdf)))
axes[1].set_title('Accuracy Ranking', fontsize=13, fontweight='bold')
axes[1].set_xlim(0.4, 1.05); axes[1].grid(axis='x', alpha=0.3)
for bar, val in zip(bars, sdf['Accuracy']):
    axes[1].text(val+0.005, bar.get_y()+bar.get_height()/2, f"{val:.4f}", va='center', fontweight='bold')

plt.suptitle('TruthLens — BERT+BiLSTM vs Traditional ML', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight'); plt.show()


## 10. Save Model

In [ ]:
import json as _json
SAVE = '/kaggle/working/truthlens_model'
os.makedirs(SAVE, exist_ok=True)
torch.save(model.state_dict(), f'{SAVE}/model_weights.pth')
tokenizer.save_pretrained(SAVE)
results_df.to_csv(f'{SAVE}/comparison.csv', index=False)
with open(f'{SAVE}/history.json','w') as f: _json.dump(history, f)
print(f"Saved to {SAVE}:")
for fn in sorted(os.listdir(SAVE)):
    print(f"  {fn}  ({os.path.getsize(os.path.join(SAVE,fn))/1e6:.2f} MB)")
